In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D2 — VINCI Consolidated Income Statement 2024
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip install pymupdf -q
from google.colab import files
from pathlib import Path

import json
import hashlib
import fitz

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D2"
DOCUMENT_NAME = "VINCI Consolidated Income Statement 2024"

BRANCH = "A"
BRANCH_NAME = "Direct Ingestion"

EXPECTED_SOURCE_FORMAT = ".pdf"
EXPECTED_PAGE_COUNT = 1
EXPECTED_RECORD_COUNT = 22

OUTPUT_DIR = Path("outputs_D2_branch_A")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FIELDS = [
    "Line Item",
    "Unit",
    "Value 2024",
    "Value 2023"
]

ALLOWED_UNITS = [
    "EUR millions",
    "EUR"
]

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. Source document upload
# ============================================================

uploaded = files.upload()

pdf_files = [
    Path(filename)
    for filename in uploaded.keys()
    if filename.lower().endswith(".pdf")
]

if len(pdf_files) != 1:
    raise ValueError(
        "Upload exactly one PDF file: the original D2 document."
    )

SOURCE_FILE = pdf_files[0]

print("Source file:", SOURCE_FILE.name)
print("Source format:", SOURCE_FILE.suffix.replace(".", "").upper())

In [ ]:
# ============================================================
# 3. Source SHA-256
# ============================================================

def calculate_sha256(file_path, chunk_size=8192):
    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(
            lambda: file.read(chunk_size),
            b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


SOURCE_SHA256 = calculate_sha256(
    SOURCE_FILE
)

print("Source SHA-256:")
print(SOURCE_SHA256)

In [ ]:
# ============================================================
# 4. Source PDF verification
# ============================================================

pdf_document = fitz.open(
    SOURCE_FILE
)

observed_page_count = len(
    pdf_document
)

diagnostic_text = "\n".join(
    page.get_text("text")
    for page in pdf_document
)

text_layer_available = bool(
    diagnostic_text.strip()
)

page_count_verified = (
    observed_page_count
    == EXPECTED_PAGE_COUNT
)

if not page_count_verified:
    raise ValueError(
        f"Unexpected D2 page count. "
        f"Expected {EXPECTED_PAGE_COUNT}, "
        f"observed {observed_page_count}."
    )

print(
    "Observed page count:",
    observed_page_count
)

print(
    "Machine-readable text layer available:",
    text_layer_available
)

print(
    "PDF source integrity verified."
)

In [ ]:
# ============================================================
# 5. Branch A representation
# ============================================================

BRANCH_REPRESENTATION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "representation_type":
        "Original source document",
    "input_file":
        SOURCE_FILE.name,
    "input_format":
        SOURCE_FILE.suffix.lower(),
    "structural_conversion_applied":
        False,
    "normalisation_applied":
        False,
    "ocr_applied":
        False,
    "derived_representation_used_as_model_input":
        False,
    "model_input_description":
        "The original PDF document is submitted directly to the LLM."
}

REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D2_branch_A_representation.json"
)

with open(
    REPRESENTATION_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        BRANCH_REPRESENTATION,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        BRANCH_REPRESENTATION,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 6. Extraction schema
# ============================================================

EXTRACTION_SCHEMA = {
    "document_id": DOCUMENT_ID,
    "record_level":
        "consolidated_income_statement_line_item",
    "fields": {
        "Line Item": {
            "type": [
                "string",
                "null"
            ],
            "description":
                "Exact visible income-statement row label"
        },
        "Unit": {
            "type": [
                "string",
                "null"
            ],
            "allowed_values":
                ALLOWED_UNITS
        },
        "Value 2024": {
            "type": [
                "number",
                "null"
            ],
            "description":
                "Reported numerical value for 2024"
        },
        "Value 2023": {
            "type": [
                "number",
                "null"
            ],
            "description":
                "Reported numerical value for 2023"
        }
    },
    "expected_output_structure": {
        "document_id":
            DOCUMENT_ID,
        "branch":
            BRANCH,
        "records": [
            {
                "Line Item": None,
                "Unit": None,
                "Value 2024": None,
                "Value 2023": None
            }
        ]
    }
}

print(
    json.dumps(
        EXTRACTION_SCHEMA,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 7. Fixed extraction task
# ============================================================

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract every line-item observation from the consolidated income
statement contained in the attached original PDF document.

Return one record for every visible income-statement line item.

For each record, extract:

- Line Item
- Unit
- Value 2024
- Value 2023

Extraction rules:

- Treat the attached original PDF as the only source of information.
- Extract only information explicitly supported by the document.
- Preserve each line-item label exactly as represented in the source,
  including footnote markers and unit text contained in the label.
- Preserve the association between each line item and its corresponding
  2024 and 2023 values.
- Use "EUR millions" for values governed by the table-level unit
  "(in € millions)".
- Use "EUR" for the two earnings-per-share observations.
- Convert financial values shown in parentheses into negative numerical
  values.
- Return Value 2024 and Value 2023 as numerical values.
- Do not calculate, infer, reconstruct, aggregate, correct or invent
  any value.
- Use null only when a requested value is not available.
- Do not include the table title, year headers, unit header or footnote
  explanation as separate records.
- Verify that every visible income-statement line item has been processed.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
"""

print(EXTRACTION_TASK)

In [ ]:
# ============================================================
# 8. Extraction prompt
# ============================================================

EXPECTED_OUTPUT_STRUCTURE = (
    EXTRACTION_SCHEMA[
        "expected_output_structure"
    ]
)

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
)}

The original PDF document is attached as the extraction source.

Return only the JSON object.
""".strip()


PROMPT_PATH = (
    OUTPUT_DIR
    / "D2_branch_A_prompt.txt"
)

PROMPT_PATH.write_text(
    FULL_PROMPT,
    encoding="utf-8"
)

print(FULL_PROMPT)
print()
print(
    "Prompt saved:",
    PROMPT_PATH
)

In [ ]:
# ============================================================
# 9. Experiment metadata
# ============================================================

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_FILE.name,

    "source_format":
        SOURCE_FILE.suffix.lower(),

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_page_count":
            EXPECTED_PAGE_COUNT,
        "observed_page_count":
            observed_page_count,
        "page_count_verified":
            page_count_verified,
        "machine_readable_text_layer":
            text_layer_available
    },

    "input_representation":
        "Original PDF document",

    "direct_document_ingestion":
        True,

    "structural_conversion_applied":
        False,

    "text_extraction_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "table_reconstruction_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_harmonisation_applied":
        False,

    "value_rounding_applied":
        False,

    "derived_calculation_applied":
        False,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,
        "expected_fields":
            EXPECTED_FIELDS
    },

    "allowed_units":
        ALLOWED_UNITS,

    "prompt_file":
        PROMPT_PATH.name,

    "expected_output_format":
        "JSON",

    "execution_environment":
        "Independent ChatGPT conversation",

    "notes": (
        "Branch A uses the original PDF directly. "
        "PDF text-layer inspection is diagnostic only "
        "and no extracted text or derived representation "
        "is supplied to the model."
    )
}

METADATA_PATH = (
    OUTPUT_DIR
    / "D2_branch_A_experiment_metadata.json"
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        EXPERIMENT_METADATA,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        EXPERIMENT_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

## Independent Branch A extraction

1. Open a new independent ChatGPT conversation.
2. Upload the original D2 PDF.
3. Submit `D2_branch_A_prompt.txt`.
4. Save the complete response exactly as returned.

In [ ]:
# ============================================================
# 10. Extraction prompt download
# ============================================================

files.download(PROMPT_PATH)

In [ ]:
# ============================================================
# 11. Raw response upload
# ============================================================

uploaded_output = files.upload()

if len(uploaded_output) != 1:
    raise ValueError(
        "Upload exactly one file containing "
        "the complete raw D2 Branch A LLM response."
    )

UPLOADED_RAW_OUTPUT_PATH = Path(
    next(iter(uploaded_output))
)

print(
    "Uploaded raw response:",
    UPLOADED_RAW_OUTPUT_PATH.name
)

In [ ]:
# ============================================================
# 12. Raw-response preservation
# ============================================================

RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D2_branch_A_raw_response.txt"
)

raw_response_text = (
    UPLOADED_RAW_OUTPUT_PATH
    .read_text(
        encoding="utf-8"
    )
)

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)

print(
    "Raw response preserved:",
    RAW_RESPONSE_PATH
)

In [ ]:
# ============================================================
# 13. Raw-response parsing
# ============================================================

valid_json = True
json_parsing_error = None
parsed_output = None

try:
    parsed_output = json.loads(
        raw_response_text
    )

except json.JSONDecodeError as error:
    valid_json = False
    json_parsing_error = str(error)

print(
    "Valid JSON:",
    valid_json
)

if json_parsing_error:
    print(
        "JSON parsing error:",
        json_parsing_error
    )

In [ ]:
# ============================================================
# 14. Parsed extraction
# ============================================================

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D2_branch_A_parsed_extraction.json"
)

if valid_json:

    with open(
        PARSED_EXTRACTION_PATH,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            parsed_output,
            file,
            indent=2,
            ensure_ascii=False
        )

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH
    )

else:

    print(
        "Parsed extraction was not created because "
        "the raw response is not valid JSON."
    )

In [ ]:
# ============================================================
# 15. Top-level structure diagnostics
# ============================================================

top_level_object_valid = False

document_id_present = False
document_id_correct = False

branch_present = False
branch_correct = False

top_level_records_present = False
records_is_list = False

records = []

if valid_json:

    top_level_object_valid = isinstance(
        parsed_output,
        dict
    )

    if top_level_object_valid:

        document_id_present = (
            "document_id"
            in parsed_output
        )

        document_id_correct = (
            parsed_output.get(
                "document_id"
            )
            == DOCUMENT_ID
        )

        branch_present = (
            "branch"
            in parsed_output
        )

        branch_correct = (
            parsed_output.get(
                "branch"
            )
            == BRANCH
        )

        top_level_records_present = (
            "records"
            in parsed_output
        )

        if top_level_records_present:

            records_is_list = isinstance(
                parsed_output[
                    "records"
                ],
                list
            )

            if records_is_list:
                records = parsed_output[
                    "records"
                ]


print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID present:",
    document_id_present
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch present:",
    branch_present
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records present:",
    top_level_records_present
)

print(
    "Records is a list:",
    records_is_list
)

print(
    "Number of records:",
    len(records)
)

In [ ]:
# ============================================================
# 16. Record-structure diagnostics
# ============================================================

record_structure_issues = []
records_with_structure_issues = 0

for record_index, record in enumerate(
    records
):

    issues = []

    if not isinstance(record, dict):

        issues.append(
            "Record is not a JSON object."
        )

    else:

        actual_fields = set(
            record.keys()
        )

        expected_fields = set(
            EXPECTED_FIELDS
        )

        missing_fields = sorted(
            expected_fields
            - actual_fields
        )

        additional_fields = sorted(
            actual_fields
            - expected_fields
        )

        if missing_fields:
            issues.append({
                "missing_fields":
                    missing_fields
            })

        if additional_fields:
            issues.append({
                "additional_fields":
                    additional_fields
            })

    if issues:

        records_with_structure_issues += 1

        record_structure_issues.append({
            "record_index":
                record_index,
            "issues":
                issues
        })


print(
    "Records with structure issues:",
    records_with_structure_issues
)

In [ ]:
# ============================================================
# 17. Field-type diagnostics
# ============================================================

field_type_issues = []
records_with_type_issues = 0

for record_index, record in enumerate(
    records
):

    if not isinstance(record, dict):
        continue

    issues = []

    line_item = record.get(
        "Line Item"
    )

    unit = record.get(
        "Unit"
    )

    value_2024 = record.get(
        "Value 2024"
    )

    value_2023 = record.get(
        "Value 2023"
    )


    if (
        line_item is not None
        and not isinstance(
            line_item,
            str
        )
    ):
        issues.append(
            "Line Item must be a string or null."
        )


    if (
        unit is not None
        and not isinstance(
            unit,
            str
        )
    ):
        issues.append(
            "Unit must be a string or null."
        )


    if (
        value_2024 is not None
        and (
            not isinstance(
                value_2024,
                (int, float)
            )
            or isinstance(
                value_2024,
                bool
            )
        )
    ):
        issues.append(
            "Value 2024 must be numerical or null."
        )


    if (
        value_2023 is not None
        and (
            not isinstance(
                value_2023,
                (int, float)
            )
            or isinstance(
                value_2023,
                bool
            )
        )
    ):
        issues.append(
            "Value 2023 must be numerical or null."
        )


    if issues:

        records_with_type_issues += 1

        field_type_issues.append({
            "record_index":
                record_index,
            "line_item":
                line_item,
            "issues":
                issues
        })


print(
    "Records with type issues:",
    records_with_type_issues
)

In [ ]:
# ============================================================
# 18. Unit diagnostics
# ============================================================

unit_issues = []

for record_index, record in enumerate(
    records
):

    if not isinstance(record, dict):
        continue

    unit = record.get(
        "Unit"
    )

    if (
        unit is not None
        and unit not in ALLOWED_UNITS
    ):

        unit_issues.append({
            "record_index":
                record_index,
            "line_item":
                record.get(
                    "Line Item"
                ),
            "observed_unit":
                unit
        })


print(
    "Records with unexpected units:",
    len(unit_issues)
)

In [ ]:
# ============================================================
# 19. Record-count and duplicate diagnostics
# ============================================================

record_count = len(records)

record_count_valid = (
    record_count
    == EXPECTED_RECORD_COUNT
)

line_item_values = [
    record.get(
        "Line Item"
    )
    for record in records
    if isinstance(
        record,
        dict
    )
]

non_null_line_items = [
    line_item
    for line_item in line_item_values
    if line_item is not None
]

duplicate_line_items = sorted({
    line_item
    for line_item
    in non_null_line_items
    if non_null_line_items.count(
        line_item
    ) > 1
})

duplicate_line_item_count = len(
    duplicate_line_items
)

print(
    "Expected records:",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records:",
    record_count
)

print(
    "Record count valid:",
    record_count_valid
)

print(
    "Duplicate line items:",
    duplicate_line_items
)

In [ ]:
# ============================================================
# 20. Missing-value diagnostics
# ============================================================

missing_values_by_field = {
    field: 0
    for field in EXPECTED_FIELDS
}

for record in records:

    if not isinstance(record, dict):
        continue

    for field in EXPECTED_FIELDS:

        if (
            field not in record
            or record.get(field) is None
        ):
            missing_values_by_field[
                field
            ] += 1


print(
    json.dumps(
        missing_values_by_field,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 21. Technical diagnostic summary
# ============================================================

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    top_level_records_present,
    records_is_list,
    records_with_structure_issues == 0,
    records_with_type_issues == 0,
    len(unit_issues) == 0
])

STRUCTURE_CHECK = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "valid_json": bool(
        valid_json
    ),
    "json_parsing_error": (
        json_parsing_error
    ),
    "top_level_object_valid": bool(
        top_level_object_valid
    ),
    "document_id_present": bool(
        document_id_present
    ),
    "document_id_correct": bool(
        document_id_correct
    ),
    "branch_present": bool(
        branch_present
    ),
    "branch_correct": bool(
        branch_correct
    ),
    "top_level_records_present": bool(
        top_level_records_present
    ),
    "records_is_list": bool(
        records_is_list
    ),
    "expected_record_count": (
        EXPECTED_RECORD_COUNT
    ),
    "number_of_records": int(
        record_count
    ),
    "record_count_valid": bool(
        record_count_valid
    ),
    "records_with_structure_issues": int(
        records_with_structure_issues
    ),
    "record_structure_issues": (
        record_structure_issues
    ),
    "records_with_type_issues": int(
        records_with_type_issues
    ),
    "field_type_issues": (
        field_type_issues
    ),
    "records_with_unexpected_units": int(
        len(unit_issues)
    ),
    "unit_issues": unit_issues,
    "duplicate_line_item_count": int(
        duplicate_line_item_count
    ),
    "duplicate_line_items": (
        duplicate_line_items
    ),
    "missing_values_by_field": (
        missing_values_by_field
    ),
    "structurally_evaluable":
        bool(structurally_evaluable)
    }

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D2_branch_A_technical_diagnostics.json"
)

with open(
    TECHNICAL_DIAGNOSTICS_PATH,
    "w",
    encoding="utf-8"
) as file:

print(
    json.dumps(
        STRUCTURE_CHECK,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 22. Experiment summary
# ============================================================

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_FILE.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        bool(
            page_count_verified
        ),

    "input_representation":
        "Original PDF document",

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "ocr_applied":
        False,

    "json_valid":
        bool(
            valid_json
        ),

    "structurally_evaluable":
        bool(structurally_evaluable),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        int(
            record_count
        ),

    "record_count_matches":
        bool(
            record_count_valid
        ),

    "records_with_structure_issues":
        int(
            records_with_structure_issues
        ),

    "records_with_type_issues":
        int(
            records_with_type_issues
        ),

    "records_with_unexpected_units":
        int(
            len(unit_issues)
        ),

    "duplicate_line_item_count":
        int(
            duplicate_line_item_count
        ),

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        bool(
            valid_json
        ),

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, "
        "Branch A execution preservation, and technical "
        "output checks only. Reference-value agreement "
        "is evaluated in the separate Stage 4 validation."
    )
}

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D2_branch_A_experiment_summary.json"
)

with open(
    EXPERIMENT_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        EXPERIMENT_SUMMARY,
        file,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 23. Final artefact inventory
# ============================================================

generated_outputs = [
    PROMPT_PATH,
    REPRESENTATION_PATH,
    METADATA_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if valid_json:
    generated_outputs.insert(
        4,
        PARSED_EXTRACTION_PATH
    )

print(
    "Generated files:\n"
)

for output_path in generated_outputs:
    print(
        "-",
        output_path.name
    )

In [ ]:
# ============================================================
# 24. Download experiment artefacts
# ============================================================

for output_path in generated_outputs:
    files.download(output_path)